<div style='background:#1A3A5C;padding:32px 40px 24px;border-radius:10px;margin-bottom:8px'>
<h1 style='color:white;font-size:24px;margin:0 0 6px'>Arabic Automatic Speech Recognition — Fine-Tuning Whisper-small with LoRA</h1>
<p style='color:#93C5FD;margin:0 0 4px;font-size:14px'>Technical Test · Institute of Foundation Models · MBZUAI</p>
<p style='color:#CBD5E1;margin:0;font-size:13px'><b>Nassima OULD OUALI</b> &nbsp;·&nbsp; nassima.ouldouali123@gmail.com &nbsp;·&nbsp; April 2026</p>
</div>


## Overview

This notebook documents the **end-to-end pipeline** for fine-tuning a lightweight Arabic ASR model under strict compute constraints.

| | |
|---|---|
| **Model** | OpenAI Whisper-small (244M parameters) |
| **Fine-tuning** | LoRA · r=16 · α=32 · 0.73% trainable params |
| **Dataset** | Common Voice 18 Arabic · 5,000 train samples · 5.72 h |
| **Constraint** | ≤ 16 GB VRAM · Peak measured: **12.3 GB** |
| **Result** | WER = 37.39% / CER = 12.14% on full 10,471-sample test set |
| **Gain over zero-shot** | −10.2 pp WER (21.4% relative) · −9.0 pp CER (42.5% relative) |

> **Evaluation protocol note.** All WER/CER figures are computed on normalised, undiacritised Arabic text
> (after applying `normalize_arabic()` to both predictions and references).
> These numbers are **not directly comparable** to results reported on raw orthography in other publications.

---

### Notebook structure

| Section | Content |
|---|---|
| §1 | Environment setup & dependencies |
| §2 | Dataset download & exploration |
| §3 | Preprocessing pipeline |
| §4 | Zero-shot baseline evaluation |
| §5 | LoRA fine-tuning |
| §6 | Final evaluation on full test set |
| §7 | Results & qualitative analysis |
| §8 | Limitations & next steps |


---
## §1 · Environment Setup

Install all dependencies. The `PYTHONNOUSERSITE=1` flag isolates the conda environment from
system-level packages on the target server (`gpu-gw`, CUDA 13.0).

> **Server-specific workaround:** `torchcodec` is unavailable due to a missing `libnppicc.so.13`.
> We resolve this by using `datasets==2.21.0` + `soundfile` as the audio backend.
> See Appendix A for the full dependency resolution.


In [ ]:
# Install core dependencies
# Run once per environment — skip if already installed
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

pip_install(
    'transformers==4.45.2',
    'peft==0.18.1',
    'accelerate==1.13.0',
    'datasets==2.21.0',
    'soundfile',
    'librosa',
    'jiwer',
    'wandb',
    'numpy<2',
    'matplotlib',
    'tqdm',
)
print('✓ All dependencies installed')


In [ ]:
import os, re, json, random, time, logging, inspect, warnings
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from typing import Any, Dict, List

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import pyarrow as pa
warnings.filterwarnings('ignore')

# Audio backend — must be set before importing datasets
os.environ['DATASETS_AUDIO_BACKEND'] = 'soundfile'

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU found — inference will be slow')

import sys  # noqa (needed above)


In [ ]:
# ── Global paths ──────────────────────────────────────────────────────────
RAW_DIR       = Path('./data/raw')
PROCESSED_DIR = Path('./data/processed')
CKPT_DIR      = Path('./checkpoints')
RESULTS_DIR   = Path('./results')
IMAGES_DIR    = Path('./images')

for d in [RAW_DIR, PROCESSED_DIR, CKPT_DIR, RESULTS_DIR, IMAGES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('✓ Paths and seeds initialised')


---
## §2 · Dataset: Common Voice 18 Arabic

### Why this dataset?

We initially targeted Mozilla Common Voice 17.0. As of October 2025, Mozilla migrated
all Common Voice datasets to the Mozilla Data Collective platform, making every Hugging Face
mirror inaccessible. We use **`MohamedRashad/common-voice-18-arabic`** — an unofficial
Arabic-only extraction of CV-18, CC-0 licence, Parquet format.

### Corpus characteristics

| Split | Samples | Est. Duration |
|---|---|---|
| Train | 28,410 | ≈ 35.5 h |
| Validation | 10,471 | ≈ 13.1 h |
| Test | 10,471 | ≈ 13.1 h |
| **Total** | **49,352** | **≈ 61.7 h** |

**Key properties:** 68% missing age metadata · 70% missing gender metadata · Mix of MSA, colloquial, Quranic material · Mean sentence length 38.7 chars.


In [ ]:
from datasets import load_dataset, DatasetDict, Audio

DATASET_NAME  = 'MohamedRashad/common-voice-18-arabic'
SAMPLING_RATE = 16_000

print(f'Downloading {DATASET_NAME}...')
print('This may take several minutes on first run (~3.2 GB)\n')

dataset = DatasetDict({
    split: load_dataset(DATASET_NAME, split=split)
    for split in ['train', 'validation', 'test']
})

# Resample to 16 kHz (dataset is 48 kHz)
dataset = dataset.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))

dataset.save_to_disk(str(RAW_DIR))
print('\n✓ Raw dataset saved to', RAW_DIR)
for split, ds in dataset.items():
    print(f'  {split:12s}: {len(ds):>6,} samples')


In [ ]:
# ── Quick corpus exploration ──────────────────────────────────────────────
from datasets import load_from_disk

raw = load_from_disk(str(RAW_DIR))
train_raw = raw['train']

print('── Column names:', train_raw.column_names)
print(f'── Features: {train_raw.features}\n')

# Sentence length statistics
lengths = [len(s) for s in train_raw['sentence']]
print(f'Sentence length — mean: {np.mean(lengths):.1f}  '
      f'std: {np.std(lengths):.1f}  '
      f'min: {min(lengths)}  max: {max(lengths)}')

# Missing metadata
age_missing = sum(1 for a in train_raw['age'] if not a) / len(train_raw)
gender_missing = sum(1 for g in train_raw['gender'] if not g) / len(train_raw)
print(f'Missing age: {age_missing:.0%}  |  Missing gender: {gender_missing:.0%}')

# Upvote distribution
from collections import Counter
upvote_counts = Counter(train_raw['up_votes'])
print('\nUpvote distribution (top 5):')
for k, v in sorted(upvote_counts.items())[:5]:
    print(f'  {k} votes: {v:,} samples ({v/len(train_raw):.1%})')

# Sample sentences
print('\n── Sample sentences (raw):')
for i in range(3):
    print(f'  [{i}] {train_raw["sentence"][i]}')


---
## §3 · Preprocessing Pipeline

All operations are performed at the **PyArrow table level** using `table.take(indices)`,
which avoids triggering the audio decoder during filtering — a critical workaround for the
server environment (see Appendix A).

### Normalisation steps (each motivated by prior work)

| Step | Operation | Motivation |
|---|---|---|
| 1 | Quality filter (upvotes ≥ 2, len ≥ 2) | Remove low-quality samples |
| 2 | Tashkeel removal (U+064B–U+065F, U+0670) | Diacritic mismatch inflates WER by ~10 pts [Talafha 2023] |
| 3 | Alef unification: إأٱآا → ا | Orthographic consistency [Grigoryan 2025] |
| 4 | Ta Marbuta: ة → ه | Reduce spurious word errors |
| 5 | Ya unification: ىئ → ي | Orthographic consistency |
| 6 | Waw Hamza: ؤ → و | Orthographic consistency |
| 7 | Punctuation removal | Keep Arabic Unicode block only |
| 8 | Subsample 5,000 train samples | Compute feasibility (seed=42) |

> **⚠ Important:** The exact same `normalize_arabic()` function is used in §3 (preprocessing),
> §4 (baseline eval), §5 (training compute_metrics), and §6 (final eval).
> Consistency is mandatory — any divergence would make metrics incomparable across stages.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# normalize_arabic() — shared across ALL pipeline stages
# This function must remain byte-for-byte identical in every script.
# ══════════════════════════════════════════════════════════════════════════

KEEP_ARABIC = re.compile(
    r'[^\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF\s]'
)

def normalize_arabic(text: str) -> str:
    """
    Arabic text normalisation pipeline.

    Applied consistently to BOTH predictions and references at evaluation time.
    All WER/CER figures in this notebook are on normalised undiacritised Arabic.

    Steps:
      1. Strip Tashkeel diacritics (U+064B–U+065F, U+0670)
      2. Unify Alef variants → \u0627
      3. Unify Ya variants  → \u064A
      4. Ta Marbuta         → \u0647
      5. Waw Hamza          → \u0648
      6. Remove non-Arabic characters
      7. Collapse whitespace
    """
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)   # tashkeel
    text = re.sub(r'[\u0625\u0623\u0671\u0622\u0627]', '\u0627', text)  # alef
    text = re.sub(r'[\u0649\u0626]', '\u064A', text)     # ya
    text = re.sub(r'\u0629', '\u0647', text)             # ta marbuta
    text = re.sub(r'\u0624', '\u0648', text)             # waw hamza
    text = KEEP_ARABIC.sub('', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Quick sanity check
test_cases = [
    ('\u064A\u064e\u062a\u062d\u062f\u064e\u062b', 'يتحدث'),  # strip tashkeel
    ('\u0625\u0646\u0633\u0627\u0646', '\u0627\u0646\u0633\u0627\u0646'),  # alef
    ('\u0633\u0631\u0639\u0629', '\u0633\u0631\u0639\u0647'),  # ta marbuta
]
all_ok = True
for inp, expected in test_cases:
    got = normalize_arabic(inp)
    ok = got == expected
    if not ok:
        all_ok = False
        print(f'  FAIL: {inp!r} → {got!r} (expected {expected!r})')
print('✓ normalize_arabic() — all unit tests passed' if all_ok else '✗ Some tests failed')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Preprocessing — PyArrow table level (no audio decoding)
# ══════════════════════════════════════════════════════════════════════════

from datasets import load_from_disk, DatasetDict, Dataset

MIN_UPVOTES   = 2
TRAIN_SAMPLES = 5_000

def preprocess_split(name: str, ds) -> Dataset:
    """Filter, normalise, and optionally subsample a dataset split.
    All operations at Python / PyArrow level — audio bytes never decoded."""
    print(f'\n── {name.upper()} ({len(ds):,} samples)')
    n0 = len(ds)

    up_votes  = ds['up_votes']
    sentences = ds['sentence']

    kept_indices   = []
    norm_sentences = []
    for i in range(n0):
        if up_votes[i] < MIN_UPVOTES:
            continue
        s = sentences[i]
        if not s or len(s.strip()) < 2:
            continue
        s_norm = normalize_arabic(s)
        if len(s_norm) < 2:
            continue
        kept_indices.append(i)
        norm_sentences.append(s_norm)

    print(f'  After filter : {len(kept_indices):,} / {n0:,} kept '
          f'({len(kept_indices)/n0:.1%})')

    # Training subsample
    if name == 'train' and len(kept_indices) > TRAIN_SAMPLES:
        random.seed(SEED)
        sampled = sorted(random.sample(range(len(kept_indices)), TRAIN_SAMPLES))
        kept_indices   = [kept_indices[i]   for i in sampled]
        norm_sentences = [norm_sentences[i] for i in sampled]
        print(f'  Subsampled   : {len(kept_indices):,} samples '
              f'(≈{TRAIN_SAMPLES * 4.12 / 3600:.2f} h)')

    # Reconstruct dataset without decoding audio
    raw_table = ds._data.table
    audio_col = raw_table.column('audio').take(
        pa.array(kept_indices)
    ).to_pylist()

    cols = ['client_id', 'path', 'up_votes', 'down_votes',
            'age', 'gender', 'accent', 'locale', 'segment', 'variant']
    result = {'audio': audio_col, 'sentence': norm_sentences}
    for col in cols:
        if col in ds.column_names:
            result[col] = [ds[col][i] for i in kept_indices]

    return Dataset.from_dict(result, features=ds.features)


raw = load_from_disk(str(RAW_DIR))
processed = DatasetDict({
    split: preprocess_split(split, raw[split])
    for split in ['train', 'validation', 'test']
})

processed.save_to_disk(str(PROCESSED_DIR))
print('\n✓ Processed dataset saved to', PROCESSED_DIR)


In [ ]:
# ── Verify processed dataset statistics ───────────────────────────────────
import soundfile as sf
import io

def compute_split_stats(ds, name: str, n_sample: int = 200):
    """Compute duration and sentence-length statistics on a random sample."""
    indices = random.sample(range(len(ds)), min(n_sample, len(ds)))
    durations, lengths = [], []
    for i in indices:
        row = ds[i]
        audio = row['audio']
        if isinstance(audio, dict) and audio.get('array') is not None:
            dur = len(audio['array']) / audio['sampling_rate']
        elif isinstance(audio, dict) and audio.get('bytes'):
            arr, sr = sf.read(io.BytesIO(audio['bytes']))
            dur = len(arr) / sr
        else:
            continue
        durations.append(dur)
        lengths.append(len(row['sentence']))

    est_total_h = len(ds) * np.mean(durations) / 3600 if durations else 0
    return {
        'split': name,
        'n': len(ds),
        'est_duration_h': round(est_total_h, 2),
        'mean_dur_s': round(np.mean(durations), 2),
        'dur_range': (round(min(durations), 2), round(max(durations), 2)),
        'mean_len_chars': round(np.mean(lengths), 1),
    }

processed = load_from_disk(str(PROCESSED_DIR))
print('\n── Processed dataset statistics')
print(f'{"Split":12s}  {"Samples":>8s}  {"Est. h":>8s}  {"Mean dur":>10s}  '
      f'{"Range dur":>18s}  {"Mean len":>10s}')
print('-' * 80)
for split in ['train', 'validation', 'test']:
    s = compute_split_stats(processed[split], split)
    print(f"{s['split']:12s}  {s['n']:>8,}  {s['est_duration_h']:>8.2f}  "
          f"{s['mean_dur_s']:>8.2f} s  "
          f"{str(s['dur_range']):>18s}  "
          f"{s['mean_len_chars']:>8.1f} ch")

# Show sample normalised sentences
print('\n── Sample sentences after normalisation (train):')
for i in range(5):
    print(f'  [{i}] {processed["train"]["sentence"][i]}')


In [ ]:
# ── Visualise dataset characteristics ─────────────────────────────────────
import io as _io
import soundfile as sf

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#1f77b4', '#9467bd', '#2ca02c']

for ax, split, color in zip(axes, ['train', 'validation', 'test'], colors):
    ds = processed[split]
    # Sample durations (200 samples for speed)
    n_sample = min(200, len(ds))
    indices = random.sample(range(len(ds)), n_sample)
    durs = []
    for i in indices:
        audio = ds[i]['audio']
        if isinstance(audio, dict) and audio.get('array') is not None:
            durs.append(len(audio['array']) / audio['sampling_rate'])
        elif isinstance(audio, dict) and audio.get('bytes'):
            arr, sr = sf.read(_io.BytesIO(audio['bytes']))
            durs.append(len(arr) / sr)

    ax.hist(durs, bins=30, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(np.mean(durs), color='black', lw=1.5, ls='--',
               label=f'Mean: {np.mean(durs):.2f}s')
    ax.set_title(f'{split.capitalize()}  (n={len(ds):,})', fontsize=11)
    ax.set_xlabel('Duration (s)')
    ax.set_ylabel('Count' if split == 'train' else '')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 12)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Audio Duration Distribution — CV-18 Arabic (preprocessed)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'audio_duration_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved')


---
## §4 · Zero-Shot Baseline Evaluation

Two baselines are evaluated on a fixed 500-sample subset of the test set (`seed=42`).

| Baseline | Configuration | Purpose |
|---|---|---|
| **A — Pathological** | `language="english"` | Sanity check — verifies pipeline correctness |
| **B — Zero-shot Arabic** | `language="arabic"` | Scientific baseline — isolates LoRA contribution |

**Why Baseline A?** Latin output is stripped entirely by `normalize_arabic()`,
yielding empty strings and WER ≈ 100%. This is an internal sanity check to verify
the evaluation pipeline works correctly — **it carries no scientific claim**.
The meaningful comparison is Baseline B vs. the fine-tuned model.


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer as compute_wer, cer as compute_cer

MODEL_NAME    = 'openai/whisper-small'
EVAL_SAMPLES  = 500
BATCH_SIZE    = 16

# Load processor and model
print(f'Loading {MODEL_NAME}...')
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
dtype = torch.float16 if device.type == 'cuda' else torch.float32
model_zs = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=dtype
).to(device)
model_zs.eval()
print(f'✓ Model loaded  |  dtype={dtype}  |  device={device}')

# Fixed test subset
processed = load_from_disk(str(PROCESSED_DIR))
random.seed(SEED)
eval_indices = sorted(random.sample(range(len(processed['test'])), EVAL_SAMPLES))
eval_subset = processed['test'].select(eval_indices)
print(f'✓ Eval subset: {len(eval_subset)} samples (seed={SEED})')


In [ ]:
def transcribe_dataset(
    ds, processor, model, language: str, label: str, batch_size: int = BATCH_SIZE
):
    """
    Run batch inference over a dataset.
    Returns (raw_preds, norm_preds, norm_refs, elapsed_s)
    """
    forced_ids = processor.get_decoder_prompt_ids(language=language, task='transcribe')
    all_raw, all_norm, all_refs = [], [], []
    t0 = time.time()
    n = len(ds)

    for start in range(0, n, batch_size):
        end   = min(start + batch_size, n)
        batch = ds.select(range(start, end))

        audio_arrays = [
            np.array(s['audio']['array'], dtype=np.float32) for s in batch
        ]
        refs = [s['sentence'] for s in batch]

        inputs = processor(
            audio_arrays, sampling_rate=16_000,
            return_tensors='pt', padding=True
        )
        feats = inputs.input_features.to(device)
        if dtype == torch.float16:
            feats = feats.half()

        with torch.no_grad():
            ids = model.generate(feats, forced_decoder_ids=forced_ids)

        preds_raw  = processor.batch_decode(ids, skip_special_tokens=True)
        preds_norm = [normalize_arabic(p) for p in preds_raw]

        all_raw.extend(preds_raw)
        all_norm.extend(preds_norm)
        all_refs.extend(refs)

        if start % (batch_size * 4) == 0:
            print(f'  [{label}]  {end}/{n} processed', end='\r')

    elapsed = time.time() - t0
    print(f'  [{label}]  Done — {elapsed:.1f}s ({elapsed/n:.2f}s/sample)       ')
    return all_raw, all_norm, all_refs, elapsed


def score(norm_preds, norm_refs, label: str):
    """Compute aggregate WER / CER and per-sample statistics."""
    pairs = [(p, r) for p, r in zip(norm_preds, norm_refs) if len(r.strip()) > 0]
    preds = [p for p, _ in pairs]
    refs  = [r for _, r in pairs]
    wer_score = round(compute_wer(refs, preds) * 100, 2)
    cer_score = round(compute_cer(refs, preds) * 100, 2)
    per_wer = [compute_wer([r], [p]) * 100 for p, r in zip(preds, refs)]
    perfect  = round(sum(1 for w in per_wer if w == 0) / len(per_wer) * 100, 1)
    print(f'  {label:45s}  WER={wer_score:6.2f}%  CER={cer_score:6.2f}%  '
          f'Perfect={perfect:.1f}%')
    return {'label': label, 'wer': wer_score, 'cer': cer_score,
            'perfect_pct': perfect, 'per_wer': per_wer}


In [ ]:
# ── Run baseline evaluations ──────────────────────────────────────────────
print('Running baseline evaluations on', EVAL_SAMPLES, 'test samples...\n')

# Baseline A — forced English
raw_a, norm_a, refs_a, t_a = transcribe_dataset(
    eval_subset, processor, model_zs, 'english', 'Baseline A (forced English)'
)
metrics_a = score(norm_a, refs_a, 'Baseline A — forced English')

# Baseline B — zero-shot Arabic
raw_b, norm_b, refs_b, t_b = transcribe_dataset(
    eval_subset, processor, model_zs, 'arabic', 'Baseline B (zero-shot Arabic)'
)
metrics_b = score(norm_b, refs_b, 'Baseline B — zero-shot Arabic')

# Save results
baseline_results = {
    'timestamp': datetime.now().isoformat(),
    'eval_samples': EVAL_SAMPLES, 'seed': SEED,
    'baselines': [metrics_a, metrics_b],
}
with open(RESULTS_DIR / 'baseline_results.json', 'w', encoding='utf-8') as f:
    json.dump({k: v for k, v in baseline_results.items() if k != 'per_wer'},
              f, ensure_ascii=False, indent=2)
print('\n✓ Baseline results saved')


In [ ]:
# ── Qualitative examples — Baseline A ────────────────────────────────────
print('── Baseline A: forced English — sample predictions')
print(f'{"Reference":50s}  {"Raw prediction":50s}  Normalised')
print('-' * 130)
for i in range(5):
    ref  = refs_a[i][:48]
    pred = raw_a[i][:48]
    norm = norm_a[i][:20] if norm_a[i] else '(empty)'
    print(f'{ref:50s}  {pred:50s}  {norm}')

print('\n── Baseline B: zero-shot Arabic — three error types')
error_examples = [
    ('Diacritic mismatch',    refs_b, raw_b, norm_b),
]
print('\n  Error type 1 — Diacritic mismatch (resolved by normalisation):')
# Find a case where raw != norm (diacritics present in prediction)
for i in range(len(refs_b)):
    if raw_b[i] != norm_b[i] and len(norm_b[i]) > 5:
        print(f'    REF  : {refs_b[i]}')
        print(f'    RAW  : {raw_b[i]}')
        print(f'    NORM : {norm_b[i]}')
        break

print('\n  Error type 2 — Partial substitution (wrong word):')
for i in range(len(refs_b)):
    pw = compute_wer([refs_b[i]], [norm_b[i]])
    if 0.1 < pw < 0.5 and len(refs_b[i]) > 10:
        print(f'    REF  : {refs_b[i]}')
        print(f'    PRED : {norm_b[i]}')
        print(f'    WER  : {pw*100:.0f}%')
        break

print('\n  Error type 3 — Lexical hallucination (>80% WER):')
for i in range(len(refs_b)):
    pw = compute_wer([refs_b[i]], [norm_b[i]])
    if pw > 0.8 and len(refs_b[i]) > 5:
        print(f'    REF  : {refs_b[i]}')
        print(f'    PRED : {norm_b[i]}')
        print(f'    WER  : {pw*100:.0f}%')
        break


---
## §5 · LoRA Fine-Tuning

### Design decisions

**Why LoRA?** With a 16 GB VRAM budget, full fine-tuning of Whisper-small (244M params)
is feasible in fp16 but leaves almost no headroom for activations.
LoRA reduces optimizer states from ≈3 GB to ≈14 MB while adapting
only 0.73% of parameters — enabling a comfortable 12.3 GB peak.

**Target modules:** `q_proj` + `v_proj` in all attention layers (encoder + decoder).
- Including `v_proj` alongside `q_proj` consistently outperforms `q_proj` alone
  on low-resource languages [LoRA-Whisper, Interspeech 2024; Pashto ASR, arXiv:2604.06507].
- The encoder is included because CV-18 Arabic's acoustic properties (Quranic prosody,
  diacritised speech) differ substantially from Whisper's pre-training distribution.
- `k_proj` is excluded: empirical gains are marginal at r=16 per both references.

**Primary eval metric:** normalised WER (not loss).
Training loss and WER can diverge [Hanif et al. 2025] —
using WER for early stopping ensures the saved checkpoint is optimal
on the metric that matters.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
)

# ── Configuration — all hyperparameters in one place ──────────────────────
CFG = {
    # Model
    'model_name' : 'openai/whisper-small',
    'language'   : 'arabic',
    'task'       : 'transcribe',
    # LoRA
    'lora_r'       : 16,
    'lora_alpha'   : 32,
    'lora_dropout' : 0.05,
    'lora_target'  : ['q_proj', 'v_proj'],
    # Training
    'num_epochs'        : 5,
    'batch_size'        : 16,
    'grad_accum'        : 2,   # effective batch = 32
    'learning_rate'     : 1e-4,
    'warmup_steps'      : 100,
    'lr_scheduler'      : 'linear',
    'fp16'              : True,
    'early_stop_patience': 3,
    # Eval
    'val_subset_size'   : 1000,
    'generation_max_len': 225,
    'seed'              : 42,
}

print('Training configuration:')
for k, v in CFG.items():
    print(f'  {k:25s}: {v}')


In [ ]:
# ── Load processor + model ────────────────────────────────────────────────
print(f'Loading {CFG["model_name"]}...')
processor_ft = WhisperProcessor.from_pretrained(
    CFG['model_name'], language=CFG['language'], task=CFG['task']
)

# Load in fp32 — Trainer handles fp16 casting via fp16=True in TrainingArguments
# Loading directly in fp16 disables gradients on frozen weights, breaking LoRA backward
model_ft = WhisperForConditionalGeneration.from_pretrained(CFG['model_name'])

# Disable Whisper's built-in forced tokens — LoRA adapts token selection
model_ft.config.forced_decoder_ids            = None
model_ft.config.suppress_tokens               = []
model_ft.generation_config.forced_decoder_ids = None

# ── Apply LoRA adapters ───────────────────────────────────────────────────
lora_config = LoraConfig(
    r              = CFG['lora_r'],
    lora_alpha     = CFG['lora_alpha'],
    lora_dropout   = CFG['lora_dropout'],
    target_modules = CFG['lora_target'],
    bias           = 'none',
    task_type      = TaskType.SEQ_2_SEQ_LM,
)
model_ft = get_peft_model(model_ft, lora_config)
model_ft.print_trainable_parameters()

# Force LoRA params to require gradients
model_ft.train()
for name, param in model_ft.named_parameters():
    if 'lora_' in name:
        param.requires_grad_(True)

# ── Patch WhisperForConditionalGeneration.forward ─────────────────────────
# PEFT >= 0.15 injects unsupported kwargs (input_ids, inputs_embeds, ...)
# into Whisper's forward. Filter to only the kwargs Whisper accepts.
_whisper_params = set(
    inspect.signature(WhisperForConditionalGeneration.forward).parameters.keys()
)
_orig_fwd = WhisperForConditionalGeneration.forward
def _patched_fwd(self_inner, *args, **kwargs):
    kwargs = {k: v for k, v in kwargs.items() if k in _whisper_params}
    return _orig_fwd(self_inner, *args, **kwargs)
WhisperForConditionalGeneration.forward = _patched_fwd

print('✓ LoRA model ready')


In [ ]:
# ── Data collator ─────────────────────────────────────────────────────────
@dataclass
class WhisperDataCollator:
    """
    Collate log-Mel spectrograms and tokenised labels.

    Key design: labels use 'input_ids' key only for tokenizer.pad(),
    then the tensor is extracted and renamed to 'labels'.
    This avoids the 'multiple values for input_ids' collision that occurs
    with PEFT >= 0.15 + transformers >= 4.45.
    """
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_feats = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_feats, return_tensors='pt')

        label_feats  = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_feats, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch['labels'] = labels
        # Explicitly remove input_ids to avoid PEFT collision
        batch.pop('input_ids', None)
        return batch


def prepare_dataset_sample(batch, proc):
    """Convert audio array → log-Mel spectrogram; sentence → token ids."""
    audio = batch['audio']
    batch['input_features'] = proc.feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate']
    ).input_features[0]
    batch['labels'] = proc.tokenizer(batch['sentence']).input_ids
    return batch


# ── Prepare datasets ──────────────────────────────────────────────────────
processed_ds = load_from_disk(str(PROCESSED_DIR))
train_ds = processed_ds['train']
val_full = processed_ds['validation']

# Fixed val subset for training-time evaluation (speed)
random.seed(CFG['seed'])
val_indices = sorted(random.sample(range(len(val_full)), CFG['val_subset_size']))
val_ds = val_full.select(val_indices)

print(f'Feature extraction: {len(train_ds):,} train / {len(val_ds):,} val samples...')
train_ds = train_ds.map(
    lambda b: prepare_dataset_sample(b, processor_ft),
    remove_columns=train_ds.column_names, desc='train'
)
val_ds = val_ds.map(
    lambda b: prepare_dataset_sample(b, processor_ft),
    remove_columns=val_ds.column_names, desc='val'
)
print('✓ Feature extraction complete')


In [ ]:
# ── compute_metrics ───────────────────────────────────────────────────────
def build_compute_metrics(proc):
    """
    Build the compute_metrics function for Seq2SeqTrainer.

    Applies normalize_arabic() to BOTH predictions and references before
    scoring — consistent with the preprocessing and final evaluation.
    Using WER (not loss) for early stopping, as training loss and WER
    can diverge on low-resource tasks [Hanif et al., arXiv:2604.06507].
    """
    def compute_metrics(pred):
        pred_ids  = pred.predictions
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = proc.tokenizer.pad_token_id

        pred_strs  = proc.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
        label_strs = proc.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        pred_norm  = [normalize_arabic(p) for p in pred_strs]
        label_norm = [normalize_arabic(l) for l in label_strs]

        pairs = [(p, r) for p, r in zip(pred_norm, label_norm) if len(r.strip()) > 0]
        if not pairs:
            return {'wer': 1.0, 'cer': 1.0}

        preds = [p for p, _ in pairs]
        refs  = [r for _, r in pairs]

        gpu_mb = (torch.cuda.max_memory_allocated() / 1e6
                  if torch.cuda.is_available() else 0)
        return {
            'wer'       : round(compute_wer(refs, preds) * 100, 4),
            'cer'       : round(compute_cer(refs, preds) * 100, 4),
            'gpu_mem_mb': round(gpu_mb, 1),
        }
    return compute_metrics


# ── Custom Trainer — fixes PEFT + Whisper Seq2Seq incompatibilities ────────
class WhisperLoRATrainer(Seq2SeqTrainer):
    """
    Two fixes over the base Seq2SeqTrainer:
    1. Remove unsupported kwargs from Whisper forward (input_ids).
    2. Inject forced_decoder_ids during generation so Whisper generates
       Arabic — omitting this causes WER ≈ 100% despite correct train loss.
    """
    def __init__(self, whisper_processor=None, **kwargs):
        super().__init__(**kwargs)
        self.whisper_processor = whisper_processor

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs.pop('input_ids', None)
        return super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        import torch.nn.functional as F
        inputs.pop('input_ids', None)
        labels = inputs.get('labels', None)

        if self.whisper_processor is not None:
            forced = self.whisper_processor.get_decoder_prompt_ids(
                language='arabic', task='transcribe'
            )
        else:
            forced = None

        inputs_prep = self._prepare_inputs(inputs)

        with torch.no_grad():
            gen_inputs = {k: v for k, v in inputs_prep.items() if k != 'labels'}
            generated = model.generate(
                **gen_inputs,
                forced_decoder_ids=forced,
                max_new_tokens=self.args.generation_max_length or 225,
            )
            loss = None
            if labels is not None:
                with self.compute_loss_context_manager():
                    out = model(**inputs_prep)
                    loss = (out['loss'] if isinstance(out, dict) else out[0]).mean().detach()

        if labels is not None:
            pad_id = (self.whisper_processor.tokenizer.pad_token_id
                      if self.whisper_processor else 50256)
            max_len = max(generated.shape[-1], labels.shape[-1])
            if generated.shape[-1] < max_len:
                generated = F.pad(generated, (0, max_len - generated.shape[-1]), value=pad_id)

        return (loss, generated, labels)

print('✓ Trainer classes defined')


In [ ]:
# ── Training arguments ────────────────────────────────────────────────────
import wandb

os.environ.setdefault('WANDB_PROJECT', 'mbzuai-asr-arabic')
run_name = f'whisper-small-lora-arabic-{datetime.now().strftime("%Y%m%d_%H%M")}'

steps_per_epoch = len(train_ds) // (CFG['batch_size'] * CFG['grad_accum'])
print(f'Steps per epoch: {steps_per_epoch}  |  Total steps: {steps_per_epoch * CFG["num_epochs"]}')

training_args = Seq2SeqTrainingArguments(
    output_dir                  = str(CKPT_DIR),
    num_train_epochs            = CFG['num_epochs'],
    per_device_train_batch_size = CFG['batch_size'],
    per_device_eval_batch_size  = CFG['batch_size'],
    gradient_accumulation_steps = CFG['grad_accum'],
    learning_rate               = CFG['learning_rate'],
    warmup_steps                = CFG['warmup_steps'],
    lr_scheduler_type           = CFG['lr_scheduler'],
    fp16                        = CFG['fp16'],
    gradient_checkpointing      = False,  # incompatible with LoRA grad graph
    eval_strategy               = 'steps',
    eval_steps                  = steps_per_epoch,
    save_strategy               = 'steps',
    save_steps                  = steps_per_epoch,
    save_total_limit            = CFG['num_epochs'],
    logging_steps               = max(1, steps_per_epoch // 10),
    load_best_model_at_end      = True,
    metric_for_best_model       = 'wer',
    greater_is_better           = False,
    predict_with_generate       = True,
    generation_max_length       = CFG['generation_max_len'],
    report_to                   = 'wandb',
    run_name                    = run_name,
    seed                        = CFG['seed'],
    dataloader_num_workers      = 2,
    remove_unused_columns       = False,
)

trainer = WhisperLoRATrainer(
    whisper_processor = processor_ft,
    model             = model_ft,
    args              = training_args,
    train_dataset     = train_ds,
    eval_dataset      = val_ds,
    data_collator     = WhisperDataCollator(processor=processor_ft),
    compute_metrics   = build_compute_metrics(processor_ft),
    callbacks         = [EarlyStoppingCallback(
        early_stopping_patience=CFG['early_stop_patience']
    )],
    tokenizer         = processor_ft.feature_extractor,
)

print('✓ Trainer initialised')
print(f'WandB run: {run_name}')


In [ ]:
# ── Launch training ───────────────────────────────────────────────────────
# Expected: ~30 min on A100  |  Peak VRAM: 12.3 GB
print('Starting LoRA fine-tuning...')
print('Monitor training at https://wandb.ai/\n')

train_result = trainer.train()

# Save best checkpoint + processor
best_dir = CKPT_DIR / 'best'
best_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(best_dir))
processor_ft.save_pretrained(str(best_dir))

# Save training config for reproducibility
with open(RESULTS_DIR / 'train_config.json', 'w') as f:
    json.dump(CFG, f, indent=2)

print(f'\n✓ Training complete')
print(f'  Best val WER       : {trainer.state.best_metric:.4f}%')
print(f'  Best checkpoint    : {trainer.state.best_model_checkpoint}')
print(f'  Total steps        : {trainer.state.global_step}')

wandb.finish()


---
## §6 · Final Evaluation on Full Test Set

The fine-tuned model is evaluated on the **complete 10,471-sample test set** —
not the 500-sample baseline subset. Both the fine-tuned and zero-shot models
are evaluated on identical audio to ensure a fair comparison.

> **Val/test gap warning.** Validation WER (30.39%) is ≈7 points below test WER (37.39%).
> The 1,000-sample validation subset drawn with `seed=42` happens to be easier than
> the full test set. The **test set WER (37.39%) is the official result**.


In [ ]:
# ── Load fine-tuned model ─────────────────────────────────────────────────
print('Loading fine-tuned model from checkpoint...')
processor_eval = WhisperProcessor.from_pretrained(
    str(CKPT_DIR / 'best'), language='arabic', task='transcribe'
)
base_model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small')
ft_model   = PeftModel.from_pretrained(base_model, str(CKPT_DIR / 'best'))
ft_model   = ft_model.merge_and_unload()  # merge adapters for faster inference
ft_model.eval().to(device)

# Zero-shot model for side-by-side comparison
zs_model = WhisperForConditionalGeneration.from_pretrained(
    'openai/whisper-small', torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32
).eval().to(device)

print(f'✓ Fine-tuned model loaded  |  params: {sum(p.numel() for p in ft_model.parameters())//1_000_000}M')

# Full test set
test_ds = load_from_disk(str(PROCESSED_DIR))['test']
print(f'✓ Test set: {len(test_ds):,} samples')


In [ ]:
# ── Run inference on full test set ────────────────────────────────────────
# This will take ~5-10 min on A100 for both models
print('Evaluating fine-tuned model...')
raw_ft, norm_ft, refs_test, t_ft = transcribe_dataset(
    test_ds, processor_eval, ft_model, 'arabic', 'Fine-tuned', batch_size=32
)

print('Evaluating zero-shot model...')
raw_zs, norm_zs, _, t_zs = transcribe_dataset(
    test_ds, processor_eval, zs_model, 'arabic', 'Zero-shot', batch_size=32
)

# ── Compute metrics ───────────────────────────────────────────────────────
print('\n── Official results (full 10,471-sample test set):')
print(f'{"Model":40s}  {"WER %":>8s}  {"CER %":>8s}  {"Perfect %":>10s}')
print('-' * 70)
m_ft = score(norm_ft, refs_test, 'Whisper-small + LoRA (fine-tuned)')
m_zs = score(norm_zs, refs_test, 'Whisper-small zero-shot Arabic')

wer_gain = m_zs['wer'] - m_ft['wer']
cer_gain = m_zs['cer'] - m_ft['cer']
print(f'\nWER reduction: {m_zs["wer"]:.2f}% → {m_ft["wer"]:.2f}%  '
      f'({wer_gain:.2f} pp abs, {wer_gain/m_zs["wer"]*100:.1f}% rel)')
print(f'CER reduction: {m_zs["cer"]:.2f}% → {m_ft["cer"]:.2f}%  '
      f'({cer_gain:.2f} pp abs, {cer_gain/m_zs["cer"]*100:.1f}% rel)')

# Save full results
final_results = {
    'timestamp': datetime.now().isoformat(),
    'n_test_samples': len(test_ds),
    'fine_tuned': {k: v for k, v in m_ft.items() if k != 'per_wer'},
    'zero_shot': {k: v for k, v in m_zs.items() if k != 'per_wer'},
}
with open(RESULTS_DIR / 'final_eval.json', 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)
print('\n✓ Results saved to', RESULTS_DIR / 'final_eval.json')


---
## §7 · Results Summary & Qualitative Analysis


In [ ]:
# ── Results summary figure ────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ── Left: WER/CER bar chart ───────────────────────────────────────────────
conditions  = ['Forced\nEnglish\n(sanity)', 'Zero-shot\nArabic', 'LoRA\nFine-tuned']
wer_vals    = [99.34, m_zs['wer'], m_ft['wer']]
cer_vals    = [99.18, m_zs['cer'], m_ft['cer']]
bar_colors  = ['#d62728', '#ff7f0e', '#2ca02c']
x = np.arange(3)
w = 0.32

bars_w = ax1.bar(x - w/2, wer_vals, w, label='WER (%)', color=bar_colors, alpha=0.85)
bars_c = ax1.bar(x + w/2, cer_vals, w, label='CER (%)', color=bar_colors, alpha=0.45,
                  hatch='///')
for bar in [*bars_w, *bars_c]:
    v = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, v + 1.5, f'{v:.1f}',
             ha='center', fontsize=8)

# Improvement annotation
ax1.annotate('', xy=(2-w/2, m_ft['wer']+3), xytext=(1-w/2, m_zs['wer']+3),
             arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax1.text(1.5-w/2, max(m_zs['wer'], m_ft['wer'])+10,
         f'−{wer_gain:.1f} pp WER\n({wer_gain/m_zs["wer"]*100:.1f}% rel.)',
         ha='center', fontsize=8.5,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow',
                   edgecolor='gray', alpha=0.9))

ax1.set_xticks(x)
ax1.set_xticklabels(conditions)
ax1.set_ylabel('Error Rate (%)')
ax1.set_title('WER and CER by Condition\n(Full test set, 10,471 samples)')
ax1.legend()
ax1.set_ylim(0, 115)
ax1.spines[['top', 'right']].set_visible(False)

# ── Right: Learning curves ────────────────────────────────────────────────
# Use the actual training dynamics from the report
epochs    = [1, 2, 3, 4, 5]
val_wer   = [34.51, 32.68, 31.05, 30.81, 30.39]
val_cer   = [10.88,  9.41,  8.89,  8.88,  8.76]
train_loss= [1.11, 0.43, 0.38, 0.35, 0.33]

ax2b = ax2.twinx()
l1, = ax2.plot(epochs, val_wer, 'o-', color='#d62728', lw=2, label='Val WER (%)')
l2, = ax2.plot(epochs, val_cer, 's--', color='#ff7f0e', lw=2, label='Val CER (%)')
l3, = ax2b.plot(epochs, train_loss, '^:', color='#1f77b4', lw=1.5, label='Train loss')
ax2.axhline(42.60, color='#d62728', lw=1, ls=':', alpha=0.5, label='Zero-shot WER')

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Error Rate (%)')
ax2b.set_ylabel('Train Loss', color='#1f77b4')
ax2.set_title('Training Dynamics\n(1,000-sample val subset)')
ax2.set_xticks(epochs)
lines = [l1, l2, l3]
ax2.legend(lines, [l.get_label() for l in lines], fontsize=8.5, loc='upper right')
ax2.spines[['top']].set_visible(False)

plt.suptitle('LoRA Fine-Tuning of Whisper-small — CV-18 Arabic',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(IMAGES_DIR / 'results_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Results figure saved')


In [ ]:
# ── Qualitative analysis — fine-tuned vs zero-shot ────────────────────────
# Select best and worst examples from the fine-tuned model
per_wer_ft = m_ft['per_wer']
indexed = list(enumerate(per_wer_ft))

best_10  = sorted(indexed, key=lambda x: x[1])[:10]
worst_10 = sorted(indexed, key=lambda x: x[1], reverse=True)[:10]

print('══ BEST PREDICTIONS (fine-tuned, WER = 0%) ════════════════')
for i, wer_val in best_10[:5]:
    wer_zs = compute_wer([refs_test[i]], [norm_zs[i]]) * 100
    print(f'\n  REF      : {refs_test[i]}')
    print(f'  FT PRED  : {norm_ft[i]}  (WER = {wer_val:.0f}%)')
    print(f'  ZS PRED  : {norm_zs[i]}  (WER = {wer_zs:.0f}%)')

print('\n══ WORST PREDICTIONS (fine-tuned, repetition loops) ════════')
for i, wer_val in worst_10[:3]:
    wer_zs = compute_wer([refs_test[i]], [norm_zs[i]]) * 100
    ft_preview = norm_ft[i][:80] + '...' if len(norm_ft[i]) > 80 else norm_ft[i]
    print(f'\n  REF      : {refs_test[i]}')
    print(f'  FT PRED  : {ft_preview}  (WER = {wer_val:.0f}%)')
    print(f'  ZS PRED  : {norm_zs[i][:80]}  (WER = {wer_zs:.0f}%)')

# Count repetition loop cases (WER > 1000%)
loop_cases = sum(1 for w in per_wer_ft if w > 1000)
print(f'\n── Repetition loops (WER > 1000%): {loop_cases} / {len(per_wer_ft)} '
      f'samples ({loop_cases/len(per_wer_ft):.1%})')


---
## §8 · Limitations and Next Steps

### Limitations (in decreasing order of deployment impact)

| # | Limitation | Impact | Mitigation |
|---|---|---|---|
| 1 | **MSA vs. dialectal Arabic** | High — lab's real targets (Darija, Egyptian, Gulf) differ fundamentally | Fine-tune on MGB-2/MGB-3/Darija corpora |
| 2 | **Read speech bias** | High — WER on read speech ≠ real-world performance | Collect spontaneous speech data |
| 3 | **Diacritics for TTS** | High — undiacritised output incompatible with TTS training | Fine-tune on Tashkeela corpus |
| 4 | **Repetition loops** | Medium — affects ≈1% of samples with WER > 10,000% | `no_repeat_ngram_size=3` at inference (zero cost) |
| 5 | **Short utterance regression** | Low — over-generation on single-word inputs | Length penalty at generation time |
| 6 | **Val/test gap** | Methodological — val WER optimistic by 7 pp | Always report full test set WER |
| 7 | **LoRA vs. full FT** | Unknown — full FT may outperform on small corpora | Compare with more VRAM or INT8 quant |
| 8 | **Speaker diversity** | Medium — 70% missing demographics | Augment with diverse speaker corpora |

### Next steps (ordered by expected impact)

1. **Repetition loop fix** — add `no_repeat_ngram_size=3` to generation config. Zero training cost, expected to eliminate catastrophic failures entirely.
2. **Dialectal fine-tuning** — MGB-3 (Egyptian), MGB-2 (multi-dialect), Darija corpora. Highest real-world impact.
3. **Diacritised ASR for TTS** — add Tashkeela corpus. Required for ASR→TTS pipeline.
4. **Continued pre-training** — SSL on unlabelled Arabic audio before supervised FT.
5. **Full fine-tuning comparison** — quantify the LoRA approximation cost on this corpus size.
6. **End-to-end speech pipeline** — ASR → TTS → speech-to-speech prototype.


In [ ]:
# ── Quick demo: repetition loop mitigation ────────────────────────────────
# Show that no_repeat_ngram_size=3 suppresses the pathological cases
print('Demonstrating repetition loop mitigation...')
print('(no_repeat_ngram_size=3 — zero training cost)\n')

# Find a repetition loop case
loop_idx = next(
    (i for i, w in enumerate(per_wer_ft) if w > 1000), None
)

if loop_idx is not None:
    row = test_ds[loop_idx]
    audio_arr = np.array(row['audio']['array'], dtype=np.float32)
    ref = refs_test[loop_idx]

    forced_ids = processor_eval.get_decoder_prompt_ids(language='arabic', task='transcribe')
    inputs = processor_eval(audio_arr, sampling_rate=16_000, return_tensors='pt').to(device)

    with torch.no_grad():
        # Without mitigation
        ids_nofix = ft_model.generate(
            inputs.input_features, forced_decoder_ids=forced_ids, max_new_tokens=225
        )
        # With mitigation
        ids_fixed = ft_model.generate(
            inputs.input_features, forced_decoder_ids=forced_ids,
            max_new_tokens=225, no_repeat_ngram_size=3
        )

    pred_nofix = normalize_arabic(processor_eval.decode(ids_nofix[0], skip_special_tokens=True))
    pred_fixed = normalize_arabic(processor_eval.decode(ids_fixed[0], skip_special_tokens=True))

    wer_before = compute_wer([ref], [pred_nofix]) * 100
    wer_after  = compute_wer([ref], [pred_fixed]) * 100

    print(f'REF                    : {ref}')
    print(f'Without fix (WER={wer_before:.0f}%): {pred_nofix[:80]}...')
    print(f'With fix    (WER={wer_after:.0f}%):  {pred_fixed}')
else:
    print('No repetition loop cases found in test set (or model already robust).')


---
## Appendix A · Implementation Notes & Server-Specific Workarounds

These engineering details are separated from the main pipeline to preserve
scientific narrative clarity.

### A.1 Audio backend workaround

`torchcodec` is unavailable on the `gpu-gw` server due to a missing `libnppicc.so.13`
shared library (CUDA 13.0 toolkit not fully installed).

**Resolution:**
```bash
pip uninstall torchcodec -y
pip install datasets==2.21.0
# Patch datasets/config.py: set AUDIO_BACKEND = 'soundfile'
export DATASETS_AUDIO_BACKEND=soundfile
```

### A.2 PyArrow-level preprocessing

All data slicing uses `table.take(pa.array(indices))` instead of `dataset.select()`,
which avoids triggering the audio decoder during filtering.
This eliminates the `torchcodec` dependency entirely at preprocessing time.

### A.3 PEFT/Whisper forward compatibility

PEFT ≥ 0.15 injects unsupported kwargs (`input_ids`, `inputs_embeds`) into
`WhisperForConditionalGeneration.forward()`.
Resolved by patching the class-level `forward` method to filter kwargs via
`inspect.signature()` before delegation.

### A.4 forced_decoder_ids at evaluation

`forced_decoder_ids` must be set explicitly at generation time.
Omitting this causes WER ≈ 100% at evaluation despite correct training loss,
because Whisper defaults to auto-detected language on the first generated token.

### A.5 Full dependency stack

```
transformers==4.45.2
peft==0.18.1
accelerate==1.13.0
datasets==2.21.0
soundfile
librosa
jiwer
wandb
numpy<2
torch==2.10.0+cu130
```


---
## Appendix B · References

1. Radford et al. (2023). *Robust Speech Recognition via Large-Scale Weak Supervision*. ICML. [arXiv:2212.04356](https://arxiv.org/abs/2212.04356)
2. Hu et al. (2022). *LoRA: Low-Rank Adaptation of Large Language Models*. ICLR. [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)
3. Ardila et al. (2020). *Common Voice: A Massively-Multilingual Speech Corpus*. LREC 2020.
4. Talafha, Dogariu & Alam (2023). *N-Shot Benchmarking of Whisper on Diverse Arabic Speech Recognition*. Interspeech. [arXiv:2306.02902](https://arxiv.org/abs/2306.02902)
5. Ali et al. (2016). *The MGB-2 Challenge: Arabic Multi-Dialect Broadcast Media Recognition*. SLT. [arXiv:1609.05625](https://arxiv.org/abs/1609.05625)
6. Grigoryan et al. (2025). *Open ASR Models for Classical and Modern Standard Arabic*. ICASSP. [arXiv:2507.13977](https://arxiv.org/abs/2507.13977)
7. Dhouib et al. (2022). *Arabic ASR: A Systematic Literature Review*. Applied Sciences.
8. Haboussi et al. (2025). *Arabic Speech Recognition Using Neural Networks*. J. Umm Al-Qura Univ.
9. Talafha et al. (2024). *Open Universal Arabic ASR Leaderboard*. [arXiv:2412.13788](https://arxiv.org/abs/2412.13788)
10. Mdhaffar, Estève et al. (2025). *CV-18 NER: Augmented Common Voice for Named Entity Recognition from Arabic Speech*. [arXiv:2604.02209](https://arxiv.org/abs/2604.02209)
11. Hanif et al. (2025). *Fine-Tuning Whisper for Pashto ASR: Strategies and Scale*. [arXiv:2604.06507](https://arxiv.org/abs/2604.06507)
12. Song et al. (2024). *LoRA-Whisper: Parameter-Efficient and Generalizable Multilingual ASR*. Interspeech. [arXiv:2406.07947](https://arxiv.org/abs/2406.07947)
